# Rooted — Story Art (SDXL, free Colab T4)

Generates wide narrative-scene illustrations for Bible Stories using open-source **Stable Diffusion XL**, run entirely on Colab's free-tier T4 GPU — no billed API calls.

**v2 — revised after the first test batch came back inconsistent** (some
outputs read as generic fantasy digital-painting, one as a stock photo,
one rendered an armored giant as a sci-fi robot). Base SDXL's default
style is not reliably "stylized 3D animation" from prompting alone —
this version adds a community-trained LoRA that's specifically and
verifiably good at that look (200K+ downloads, 575 reviews on CivitAI),
plus stronger negative prompts and two scene-description fixes for the
specific failures seen (the robot-Goliath and the silhouette-only
Crucifixion).

**If you already ran an earlier version of this notebook and hit
`ImportError: Found an incompatible version of torchao`**: Runtime →
Restart session, then run all cells fresh from the top. The install
cell below now upgrades `torchao`/`peft` to versions `load_lora_weights`
actually needs, but a package upgrade doesn't take effect in a Python
process that already imported the old version — only a full restart
picks it up.

**If you hit `IndexError: list index out of range` in
`get_peft_kwargs`** on `pipe.load_lora_weights(...)`: the diagnostic
cells confirmed the LoRA file itself is fine (standard Kohya SDXL
naming, all three components — unet/te1/te2 — present with no
unrecognized keys); the crash was traced to the latest `diffusers`
release (0.40.0) mishandling this file. The install cell now pins
`diffusers==0.31.0` instead of always taking the newest release.

**If pinning diffusers then produces `ImportError: cannot import name
'FLAX_WEIGHTS_NAME' from 'transformers.utils'`**: that's a second,
separate version mismatch — `transformers` was still on `-U`/latest,
which dropped Flax support (and that symbol) entirely, but pinned
`diffusers==0.31.0` still expects it at import time. The install cell
now also pins `transformers==4.46.3`, from the same release era as
`diffusers 0.31.0`. **Runtime → Restart session** is required again
after either version-pin change, same reason as the torchao fix above.

**Before running:** download the LoRA file yourself first (Colab can't
fetch it with a plain URL reliably) —
1. Go to https://civitai.com/models/188525 ("Pixar Style (SDXL)").
2. Download the `.safetensors` file (~218MB).
3. Keep it ready to upload in the cell below.

**How to use:**
1. Runtime → Change runtime type → T4 GPU (free tier).
2. Run all cells top to bottom — you'll be prompted to upload the LoRA file partway through.
3. Review the inline previews in the generation cell.
4. Run the final cell to download `story_art.zip`.
5. Unzip it into `pipeline/.storyart_incoming/` in the project folder, then run:
   `py pipeline/import_story_art.py`
   which resizes/compresses each image and moves it into `media/stories/`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# peft (needed for load_lora_weights below) and its torchao dependency
# are explicitly pinned/upgraded here — Colab's preinstalled torchao
# (0.10.0 as of writing) is too old for the peft version diffusers pulls
# in otherwise, which fails with "Found an incompatible version of
# torchao" the moment load_lora_weights() runs, several cells later.
#
# diffusers is PINNED to 0.31.0 rather than left on `-U` (latest). On the
# latest release (0.40.0), load_lora_weights() crashed with "IndexError:
# list index out of range" in get_peft_kwargs() on this exact LoRA file —
# confirmed via the diagnostic cells below that the file itself is a
# completely standard Kohya SDXL LoRA (proper lora_unet_/lora_te1_/
# lora_te2_ keys, all three components present, no unrecognized keys), so
# the file isn't the problem.
#
# transformers is ALSO pinned, to the same era as diffusers 0.31.0
# (released ~Oct 2024) — pinning diffusers alone and leaving transformers
# on `-U` broke a different way: transformers' newest releases dropped
# Flax support entirely (removed `FLAX_WEIGHTS_NAME` from
# transformers.utils), but diffusers 0.31.0's SDXL pipeline still expects
# that symbol to exist at import time, so pulling latest transformers
# against pinned diffusers just traded one crash for another.
#
# If you already hit either error once in this session, Runtime →
# Restart session and run all cells again from the top — pip
# upgrading/pinning a package that's already been imported into the
# running Python process doesn't take effect until the kernel restarts.
!pip install -q "diffusers==0.31.0" "transformers==4.46.3" accelerate safetensors invisible_watermark peft "torchao>=0.16.0"

In [ ]:
import torch
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler

pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True,
)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to("cuda")
pipe.enable_attention_slicing()  # keeps this comfortably inside the T4's 16GB

In [ ]:
# Upload the "Pixar Style (SDXL)" LoRA you downloaded from
# https://civitai.com/models/188525 — a genuinely well-validated SDXL LoRA
# for exactly the look this project wants (200K+ downloads, 575 reviews),
# not a guess. Base SDXL alone did not reliably produce this style in the
# first test batch; this is the fix, not a prompt tweak.
from google.colab import files

print("Select the Pixar Style (SDXL) .safetensors file you downloaded from civitai.com/models/188525")
uploaded = files.upload()
LORA_FILENAME = next(iter(uploaded))
print(f"\nuploaded: {LORA_FILENAME} ({len(uploaded[LORA_FILENAME]) / 1e6:.1f} MB)")

In [ ]:
# Diagnostic — run this BEFORE the load cell below. The previous attempt
# failed with "IndexError: list index out of range" inside diffusers'
# get_peft_kwargs(), which means diffusers parsed the file but found ZERO
# keys matching either the "diffusers-native" or "Kohya-style" LoRA
# naming patterns it knows how to convert. This prints what's actually in
# the file so the real cause (wrong file, unexpected key prefix, a
# text-encoder-only LoRA, etc.) is visible instead of guessed at again.
from safetensors.torch import load_file

state_dict = load_file(LORA_FILENAME)
keys = list(state_dict.keys())
print(f"{len(keys)} tensor keys in {LORA_FILENAME}\n")
print("first 15 keys:")
for k in keys[:15]:
    print(" ", k)

kohya_style = any(k.startswith("lora_unet_") or k.startswith("lora_te") for k in keys)
diffusers_native = any(".lora.down.weight" in k or ".lora_A." in k or ".lora_B." in k for k in keys)
print(f"\nlooks like Kohya-style keys (lora_unet_*/lora_te*): {kohya_style}")
print(f"looks like diffusers-native keys (.lora_A./.lora_B.): {diffusers_native}")
if not kohya_style and not diffusers_native:
    print("\n⚠ Neither pattern matched — this is almost certainly why load_lora_weights failed.")
    print("  Share the printed key list above and we'll figure out the right loading call.")

In [ ]:
# Follow-up diagnostic, run after seeing the cell above's output. The
# printed keys are standard Kohya SDXL naming (lora_te1_.../lora_unet_...,
# .alpha/.lora_down.weight/.lora_up.weight) -- nothing wrong with the file
# itself. The IndexError inside get_peft_kwargs() most likely means one of
# SDXL's three LoRA-able components (unet, text_encoder = te1,
# text_encoder_2 = te2) has ZERO matching keys -- a LoRA trained on
# unet+te1 only, skipping te2 entirely, is common, and some diffusers
# versions crash instead of skipping that empty component gracefully.
# This counts keys per component so we know which case we're actually in
# before touching anything else.
import diffusers, re

unet_n = sum(1 for k in keys if k.startswith("lora_unet_"))
te1_n  = sum(1 for k in keys if k.startswith("lora_te1_"))
te2_n  = sum(1 for k in keys if k.startswith("lora_te2_"))
te_n   = sum(1 for k in keys if re.match(r"^lora_te_", k))  # single-encoder naming, if present
other_n = len(keys) - unet_n - te1_n - te2_n - te_n

print(f"diffusers version: {diffusers.__version__}")
print(f"lora_unet_* keys: {unet_n}")
print(f"lora_te1_*  keys: {te1_n}")
print(f"lora_te2_*  keys: {te2_n}")
print(f"lora_te_*   keys (single-encoder naming): {te_n}")
print(f"unrecognized-prefix keys: {other_n}")
if other_n:
    print("\nsample unrecognized keys:")
    seen = {unet_n, te1_n, te2_n, te_n}
    shown = 0
    for k in keys:
        if not (k.startswith("lora_unet_") or k.startswith("lora_te1_") or k.startswith("lora_te2_") or re.match(r"^lora_te_", k)):
            print(" ", k)
            shown += 1
            if shown >= 10:
                break
if te2_n == 0:
    print("\n-> te2 has zero keys. This LoRA doesn't touch SDXL's second text encoder at")
    print("   all, which is a very likely match for the IndexError. Fix: upgrade diffusers")
    print("   (pip install -U diffusers) and restart the runtime, since this exact")
    print("   empty-component crash was hardened against in more recent diffusers releases.")


In [ ]:
LORA_SCALE = 0.9  # the LoRA card's own recommended range is roughly 0.8-1.0

try:
    pipe.load_lora_weights(LORA_FILENAME)
    print(f"loaded LoRA: {LORA_FILENAME} @ scale {LORA_SCALE}")
except IndexError as e:
    # This is the specific failure this cell has already hit once — surface
    # the diagnostic-cell findings right here instead of just the raw
    # traceback, so the next step is obvious without re-reading everything
    # above.
    print("load_lora_weights failed:", e)
    print(f"Kohya-style keys detected: {kohya_style} | diffusers-native keys detected: {diffusers_native}")
    if not kohya_style and not diffusers_native:
        print("Neither key pattern matched in the diagnostic cell above — this file's internal")
        print("structure isn't one diffusers' loader recognizes. Re-download from")
        print("civitai.com/models/188525 (the download may have been incomplete or the wrong")
        print("file variant) and re-run from the upload cell, or share the printed key list so")
        print("the loading call can be adjusted to match.")
    raise

In [ ]:
# Same design language as pipeline/generate_character_art.py's STYLE_SUFFIX,
# adapted for a wide multi-figure narrative SCENE rather than a close-up
# single-character portrait. Kept as its own constant (not shared code)
# since this runs in a completely separate environment (Colab, not the
# local pipeline) with a different model (SDXL, not Gemini). "pixar style"
# is the LoRA's own trigger phrase — the card's docs say to include it
# literally in the prompt for the style to activate reliably.
STYLE_SUFFIX = (
    "pixar style, richly rendered stylized 3D character-animation art, in "
    "the polished feel of a modern animated feature film. Warm, painterly "
    "cinematic lighting; expressive, dignified character posing; soft, "
    "tactile shading on skin, hair, and fabric. Ancient Near Eastern "
    "biblical-era clothing and setting rendered respectfully and without "
    "caricature. Wide establishing scene, full figures and environment "
    "both clearly visible, full color, cinematic framing."
)

# Expanded after the first test batch's specific failures: generic
# "digital painting"/"concept art" looks (Creation, the Flood), a
# monochrome stock-photo silhouette (the Crucifixion), and mechanical/
# robotic anatomy where a human giant was intended (Goliath). SDXL's
# negative-prompt channel is the right place for all of this — negation
# embedded in the main positive prompt (as generate_character_art.py does
# successfully for Gemini) is much less reliable for SD-family models.
NEGATIVE_PROMPT = (
    "photorealistic, photograph, realistic skin texture, film grain, DSLR, "
    "live action, real human face, hyperrealistic, digital painting, concept "
    "art, matte painting, stock photo, flat vector art, 2D cartoon, anime, "
    "monochrome, black and white, silhouette only, underexposed, low "
    "contrast, robot, mecha, mechanical, cyborg, sci-fi armor, low quality, "
    "blurry, deformed, extra limbs, extra fingers, mutated hands, watermark, "
    "text, signature, logo, ugly, disfigured, close-up, portrait"
)

def build_prompt(title, reference, scene):
    return f"{title} ({reference}): {scene}. {STYLE_SUFFIX}"

In [ ]:
# Test batch — matches pipeline/curation/story_art_settings.json in the
# repo (also updated for david_and_goliath/crucifixion, see below). For
# the full ~324-story run later, replace this dict with one loaded from
# an uploaded story_art_settings.json, keeping this same shape.
STORIES = {
    "story_creation": {
        "title": "The world is made",
        "reference": "Genesis 1",
        "scene": "God's Spirit hovering over a vast cosmic void of swirling light and darkness, the very first moment of creation, stars and galaxies just beginning to form in the distance",
    },
    "story_the_flood": {
        "title": "The flood",
        "reference": "Genesis 6-9",
        "scene": "Noah's massive wooden ark floating on a vast flooded world under a breaking stormy sky, distant mountain peaks barely visible above the waterline, a single beam of light breaking through the clouds",
    },
    "story_feeding_the_5000": {
        "title": "Feeding the five thousand",
        "reference": "Matthew 14; Mark 6; Luke 9; John 6",
        "scene": "Jesus and his disciples distributing bread and fish among a vast seated crowd on a grassy hillside at golden hour, overflowing baskets of food being carried through the crowd",
    },
    "story_david_and_goliath": {
        "title": "David and Goliath",
        "reference": "1 Samuel 17",
        # Fixed: the original wording ("massive armored... giant") rendered
        # as a sci-fi robot. Now explicit that Goliath is a fully human man,
        # just very large, wearing bronze/leather armor.
        "scene": "young David facing Goliath, a giant of a MAN (fully human, not a robot or machine) wearing bronze scale armor and a leather kilt, across the battlefield of the valley of Elah, a sling raised in David's hand, both armies watching tensely from either side",
    },
    "story_crucifixion": {
        "title": "The crucifixion",
        "reference": "Matthew 27; Mark 15; Luke 23; John 19",
        # Fixed: the original wording ("silhouetted") produced a monochrome
        # stock-photo look with no illustrated character detail at all. Now
        # asks for lit, colored figures instead of pure silhouette.
        "scene": "three crosses on a hill outside Jerusalem at dusk, warm dramatic sunset light illuminating the mourners' faces and clothing in full color (not pure black silhouettes), a small group gathered at a respectful distance below, a darkened stormy sky overhead",
    },
    "story_lions_den": {
        "title": "Daniel in the lions' den",
        "reference": "Daniel 6",
        "scene": "Daniel, his face clearly lit and visible, standing calm and unharmed among several realistic lions (not dogs or other animals) in a torchlit stone den, soldiers and the king peering down anxiously from an opening high above",
    },
}

In [ ]:
import os
from IPython.display import display

OUT_DIR = "/content/story_art"
os.makedirs(OUT_DIR, exist_ok=True)

# 1216x832 is one of SDXL's native trained resolutions (~3:2 landscape) —
# suits a wide multi-figure scene far better than a square portrait crop.
for story_id, info in STORIES.items():
    prompt = build_prompt(info["title"], info["reference"], info["scene"])
    print(f"--- {story_id} ---")
    print(prompt)
    image = pipe(
        prompt=prompt,
        negative_prompt=NEGATIVE_PROMPT,
        height=832,
        width=1216,
        num_inference_steps=30,
        guidance_scale=7.0,
        cross_attention_kwargs={"scale": LORA_SCALE},
    ).images[0]
    out_path = os.path.join(OUT_DIR, f"{story_id}.png")
    image.save(out_path)
    display(image)
    print(f"saved -> {out_path}\n")

In [ ]:
import shutil
from google.colab import files

zip_path = shutil.make_archive("/content/story_art", "zip", OUT_DIR)
files.download(zip_path)